# 보스톤 집값 예측

## 데이터 불러오기

In [2]:
# 파이썬 경고 무시 설정
import warnings
warnings.filterwarnings('ignore')

In [4]:
# 필요한 라이브러리 임포트
import pandas as pd

# 파일 경로 설정
file_path='C:/Users/playdata2/work/house_price.csv'

# DataFrame 생성
df = pd.read_csv(file_path)

# 결과 확인
display(df)

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


## 학습용과 평가용 데이터 생성

In [6]:
# 75:25의 비율로 학습용과 평가용 데이터 생성

## 필요한 함수 임포트
from sklearn.model_selection import train_test_split

## X_data 생성: 정답 컬럼 제거
X_data = df.drop(columns=['medv'])
display(X_data)

print('-'*80)

## y_data 생성: 정답 컬럼 인덱싱
y_data = df['medv']
print(y_data)

print('-'*80)

## train_test_split() 함수 호출 -> 무작위성 통제 -> random_state
X_train, X_test, y_train, y_test = train_test_split(
    X_data,
    y_data,
    test_size=0.25,
    random_state=0
)

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,9.67
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48


--------------------------------------------------------------------------------
0      24.0
1      21.6
2      34.7
3      33.4
4      36.2
       ... 
501    22.4
502    20.6
503    23.9
504    22.0
505    11.9
Name: medv, Length: 506, dtype: float64
--------------------------------------------------------------------------------


In [7]:
# 학습용 데이터의 인덱스 확인

## X_train의 인덱스 확인
print(X_train.index)

print('-'*80)

## y_train의 인덱스 확인
print(y_train.index)

Index([245,  59, 276, 395, 416, 491, 170, 419,  55, 235,
       ...
       211,   9, 359, 195, 251, 323, 192, 117,  47, 172],
      dtype='int64', length=379)
--------------------------------------------------------------------------------
Index([245,  59, 276, 395, 416, 491, 170, 419,  55, 235,
       ...
       211,   9, 359, 195, 251, 323, 192, 117,  47, 172],
      dtype='int64', length=379)


## DecisionTree 회귀 모델 고도화

### 탐색 도구 생성 및 탐색 실행

In [8]:
# 필요한 클래스 임포트
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

# 성능 고도화 대상 모델 생성
dt = DecisionTreeRegressor(random_state=0)

# 최적화 할 매개 변수와 탐색 조건 설정 -> 18 * 4 = 72가지 조건 탐색
params = {
    'max_depth': range(3,21),
    'min_samples_leaf': [2,3,4,5]
}

# 탐색 도구 생성
grid_dt = GridSearchCV(
    estimator=dt,
    param_grid=params,
    scoring='neg_root_mean_squared_error',
    cv=5
)

# 탐색 도구 학습 및 검증
grid_dt.fit(X_train, y_train)
print("검증 완료")

검증 완료


In [9]:
# best 모델 추출
best_dt = grid_dt.best_estimator_
print(best_dt)

DecisionTreeRegressor(max_depth=8, min_samples_leaf=5, random_state=0)


In [10]:
# 검증용 데이터에 대한 최적의 성능 확인
print(grid_dt.best_score_)

-3.8549378031198294


### 평가용 데이터에 대한 예측 

In [12]:
pred_dt = best_dt.predict(X_test)
print(pred_dt)
print('-'*80)
print(f'평가용 데이터의 정답(실제 주택 가격): \n{y_test}')

[24.18235294 20.28       23.03333333 12.22857143 21.26       20.65789474
 20.65789474 19.2125     22.74       23.03333333 11.0125     13.82
 17.11666667  6.58333333 44.9        34.66315789 21.26       34.66315789
 24.18235294 20.65789474 24.18235294 20.275      20.275      25.78
 20.65789474 17.7        17.9        15.3        39.5875     19.78
 17.11666667 20.275      16.58       23.44444444 23.03333333 20.275
  8.3625     20.28       12.97777778 13.82       23.44444444 20.65789474
 21.26       17.11666667 26.86       17.9        21.58       15.3
 15.38888889 24.325      15.3        16.74       20.65789474 39.5875
 15.3        18.5625     20.65789474 19.2125     20.28       21.58
 21.175      23.44444444 34.66315789 24.18235294 20.65789474 24.18235294
 16.74       18.5625     12.22857143 20.65789474 16.58       21.26
 23.85       32.22222222 24.8         6.58333333 39.5875     23.44444444
 22.02222222 20.65789474 27.12222222 20.275      17.7        39.5875
 39.5875     24.18235294 23.

### 모델 평가: RMSE

In [13]:
# 필요한 함수 임포트
from sklearn.metrics import root_mean_squared_error as rmse

# 평가용 데이터에 대한 정답(주택 가격)과의 오차(RMSE) 측정
rmse_dt = rmse(y_test, pred_dt)

# 결과 출력
print(f'고도화된 DecisionTree 회귀 모델의 성능(RMSE): {rmse_dt:.3f}')

고도화된 DecisionTree 회귀 모델의 성능(RMSE): 5.398


## RandomForest 회귀 모델 고도화

### 탐색 도구 생성 및 탐색 실행

In [ ]:
# 필요한 클래스 임포트
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# 성능 고도화 대상 모델 생성
rf = RandomForestRegressor(random_state=0)

# 최적화 할 매개 변수와 탐색 조건 설정: 탐색 조건=560가지
params = {
    'n_estimators':[100,200,300,400,500],
    'max_depth':range(3,31),
    'min_samples_leaf':[2,3,4,5]
}

# 탐색 도구 생성
random_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=params,
    n_iter=120,
    scoring='neg_root_mean_squared_error',    
    cv=5,
    random_state=0
)

# 학습 및 검증
random_rf.fit(X_train, y_train)
print("검증 완료")

검증 완료


In [28]:
# best 모델 추출
best_rf = random_rf.best_estimator_
print(best_rf)

RandomForestRegressor(max_depth=10, min_samples_leaf=2, random_state=0)


In [30]:
# 검증용 데이터에 대한 최적의 성능(-RMSE)
print(random_rf.best_score_)

-3.4888854344274214


### 평가용 데이터에 대한 예측

In [31]:
pred_rf = best_rf.predict(X_test)
print(pred_rf)

[23.76274618 26.73144454 21.61961047 11.11344297 20.53351761 20.76307095
 21.38375523 20.37711624 20.80043157 19.49369326  8.80997731 13.77008333
 14.97979643  8.54990952 46.88851429 33.99375794 21.29713215 34.68052024
 25.63238657 21.08952668 23.7240605  21.97744683 19.64683529 24.38065446
 20.50539503 18.50650476 18.06212563 15.18097698 43.27693333 18.53240238
 14.91843489 17.4599011  20.68600069 20.87727949 23.04471983 17.24167716
  8.23896746 25.97791585 13.82058132 14.4762119  22.57356648 20.69182332
 22.45925415 15.43520394 23.27619081 22.42576282 20.6913619  15.91129127
 14.63241772 24.89512619 15.93142143 18.99095    21.65069462 38.31538929
 14.72443846 19.73635925 19.50782945 19.13118869 21.1376876  20.33900229
 21.81798899 21.1815572  33.12222778 28.4934784  18.39574304 27.16083936
 16.20251753 18.10751261 17.27299524 21.22170126 19.95593541 22.54438867
 24.30013459 32.02978889 27.11287857  8.35677143 42.21044762 21.84254885
 22.87234228 20.46649346 26.90614685 17.92891429 20

### 모델 평가

In [32]:
# 평가에 필요한 함수 임포트
from sklearn.metrics import root_mean_squared_error as rmse

# 평가용 데이터에 대한 성능(RMSE) 측정
rmse_rf = rmse(y_test, pred_rf)

# 결과 확인
print(f'고도화된 RandomForest 회귀 모델의 성능(RMSE): {rmse_rf:.3f}')

고도화된 RandomForest 회귀 모델의 성능(RMSE): 4.305
